### Judging ID-classification accuracy and OOD-generation quality.

tl;dr Quality for both, classification of whether a negative trace contains a calculation error and quality of the error-augmented traces, are quite low. From this I'm drawing three possible patches to apply and retry:

- Use stronger base model for classification/augmentation. I'll make another run, this time using Qwen3-32B.
- Prompt may be too complicated. The prompt may be too complicated. I'll reduce the complexity of the examples and re-generate the base datasets.
- Negative data includes a lot of python-hallucination answers - they seem quite frequent. I wouldn't want to use those anyways, perhaps I can pre-filter those out by removing any trace which contains "```python"
- The data seems to be quite hard, answers are quite long. It may just be that the complexity of this task is too high. I wouldn't want to switch to gsm8k now, lets retry with the above patches and see whether to escalate later.

#### 10.12

Evaluating Qwen3-8B's Old Prompt classifications:

out of 20 samples, 10 True positives, False Positives: 5, True negatives: 5, False negatives: 0.
Precision: TP / (TP + FP) = 10 / 15 = 2/3.
Accuracy: 15/20 = 0.75.

#### To make comparison fair:
- Should always sample the same answers, have to set the random_state in the df.sample operation in the ipynbs
- Should specify "algebraic or arithmetic error" instead of just "calculation error".
- Should increase response length
- Should include of list of errors distinct from "algebraic or arithmetic error" -> "wrong assumption, logic error, reasoning error..."

#### 11.12

#### Notes on ID data

Evaluating the id-outputs-simple accuracy. I've included the notes from yesterday.

I've reduced the analysis to calculating precision. With the simplified prompt, I'm getting way less positive samples, which
in itself is promising. We've got 5 TPs and 1 FP => precision is 5/6.

Especially adding to the prompt that in case of a reasoning error the LLM should return '#### no' seemed to have helped alot and the false positive rate.

#### Notes on OOD data

Evaluating the ood-outputs-simple accuracy. What I'd like to see is reasonable augmentations, the focus should again lie on a high precision in the augmentation, i.e. a ground truth which doesn't reasonably allow for augmentation with arithmetic or algebraic errors should just be skipped.

`ood-outputs-simple.parquet`:
I've noticed that alot of samples are being rejected - in some cases even because the error would've been too forced which is amazing. I should think about extracting those good samples as examples to use in the next iteration of the prompts.

Other than that, there are alot of problems with the augmented responses, they're only partly solving the problem, have a format completely distinct from the original ground truth and contain "artifacts" like "... i'm going to incorrectly assume ..." or " ... (I've included an error here) ...", which introduces noise. I should think about hard-removing those sequences based on some keywords.

`ood-outputs-simple2.parquet`:
- Qw3-8b returned only the final augmented answer. Ambiguous prompt. I should provide a positive example here.


In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
with open('/u/rfechner/data/ariadne/debug-ood-outputs-simple2.parquet', 'rb') as file:
    df = pd.read_parquet(file)

In [3]:
samples = df.sample(n=20, random_state=0)

### OOD Analysis

Number of samples: 20
Number of degenerate samples: 20

### ID-Analysis
Measure the accuracy  of the classification.
Number of samples: 20
Number of correct classifications: 1 (debatable whether this is a spurious success)
Number of degenerate/misunderstood answers: 19

In [13]:
i = 9
print(samples.iloc[i]['prompt'][1]['content'])
print("\n\n######################\n\n", samples.iloc[i]['responses'][0])

You're given a question and a correct student answer, your task is to inject an arithmetic or algebraic error (e.g., adding, subtracting, multiplying, or simplifying incorrectly) if and only if the answer allows for a reasonable augmentation. IMPORTANT: Stay as close as possible to the correct answer and refrain from explicitly stating errors in the augmented response, e.g. writing 'I incorretly calculate ...' or '... (this is a calculation error) ...'.In case it is unreasonable to augment the correct answer with an arithmetic or algebraic error (some answers do not contain arithmetic operations or algebraic manipulations), just return '#### Not applicable'. Otherwise return the complete error-augmented answer, pre-pended by a '####'. Abstract example: If the Correct Answer is [reasoning] [correct arithmetic step 1] ... [correct arithmetic step k] ... [correct result], then your answer should be #### [reasoning] [correct arithmetic step 1] ... [incorrect arithmetic step k] ... [incorre